# Lesson 2 - Vector Search - Code Along 5 - ONNX Runtime

Instead of using PyTorch to serve the embedding model, we use ONNX.
This significantly reduces the number of dependencies and size of environment.
During development, a large environment is fine, but for production, it's
at least desired, if not necessary to keep it minimal.

In [17]:
# dependencies
import sys
import numpy as np

from tqdm.auto import tqdm

# custom code imports

from embedder import Embedder

# ingest.py lives in 02-vector-search/, one level above this onnx subproject
# modify path to enable importing from that location
sys.path.insert(0, "..")

from ingest import load_faq_data

In [5]:
# verify the correct environment is loaded
# this must print the path to the .venv inside the llm-zoomcamp-onnx sub dir
# not the one in the llm-zoomcamp-homework base dir
print(sys.executable)

/Users/fakrueg/projects/courses/datatalks/llm-zoomcamp/llm-zoomcamp-homework/02-vector-search/llm-zoomcamp-onnx/.venv/bin/python3


In [6]:
# initialize embedder - model from ONNX weights plus some surrounding code
embed = Embedder()

# define two example questions and one example answer to work with
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

# embedd the examples
v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [10]:
# compute similarities
print(f"Similarity between q1 and d: {np.dot(v1, dv)}")
print(f"Similarity between q2 and d: {np.dot(v2, dv)}")

Similarity between q1 and d: 0.3233238326017993
Similarity between q2 and d: 0.019730417971579928


In [15]:
# load FAQ documents
documents = load_faq_data()

In [16]:
# combine question and answer for each document
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [18]:
# embed in batches
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

In [19]:
# run vector search manually
scores = X.dot(v1)
idx = np.argmax(scores)

documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}